In [ ]:
import torch
import numpy as np


from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models
from dinosaw.utils import do_2D_pca, get_features, add_custom_font

from dinosaw.utils import do_2D_pca
from skimage.transform import resize

from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
selected_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dinov3_s+', 'alibi_coco_dinov2_s', 'nope_dinov2_s')
models = get_models(selected_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")


n_dims = 384

2026-07-24 14:34:32 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 14:34:32 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-24 14:34:33 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 14:34:33 | I | factory.py                 : 152 | Building wrapper 'dinov3_s+' on device cuda:0
2026-07-24 14:34:33 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='torch_hub', model_arch='dinov3_s+', pretrained=False, checkpoint_path='../../models/checkpoints/backbones/dinov3_vits_patch16_plus_reg4.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 14:34:33 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/backbones/dinov3_vits_patch16_plus_reg4.pth
2026-07-24 14:34:33 | I | wrapper.py                 :  48 | Initialized PretrainedViTWra

In [7]:
img = Image.open("data/length_generalization/bulldog_518.png")

sizes = (518, 784, 1036, 1246)

features: dict[ModelTypes, list[np.ndarray]] = {key: [] for key in selected_models}

pca_scaling = 'std'
for key, model in models.items():
    for size in sizes:
        image = img.resize((size, size), Image.Resampling.BILINEAR)
        feats = get_features(model, image)
        reduced = do_2D_pca(feats, 3, pre_norm=pca_scaling, post_norm='minmax')[:, :, 0:9]
        features[key].append(reduced)

2026-07-24 14:35:56 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 14:35:57 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 14:35:57 | I | wrapper.py                 :  92 | Processing image, size: [784, 784]
2026-07-24 14:35:57 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,784,784] -> f: [1,384,56,56]
2026-07-24 14:35:57 | I | wrapper.py                 :  92 | Processing image, size: [1036, 1036]
2026-07-24 14:35:57 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,1036,1036] -> f: [1,384,74,74]
2026-07-24 14:35:57 | I | wrapper.py                 :  92 | Processing image, size: [1246, 1246]
2026-07-24 14:35:57 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,1246,1246] -> f: [1,384,89,89]
2026-07-24 14:35:57 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 14:35:57 | I | wrapper.py            

In [9]:
def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [20]:
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
n_rows, n_cols = len(selected_models) + 1, len(sizes)
W, H = 7.5, 2.3 * 3.5


fig, axs = plt.subplots(n_rows, n_cols, figsize=(W, H))


axs[0, 0].imshow(img)
hide_axes(axs[0, 0])
for i in range(1, n_cols):
    axs[0, i].remove()

for col, size in enumerate(sizes):
    # axs[0, col].imshow(image, rasterized=True)
    # hide_axes(axs[0, col])
    # axs[0, col].set_title(titles[col], weight=500)

    for row, key in enumerate(selected_models):
        ax = axs[row + 1, col]
        ax.imshow(features[key][col], rasterized=True)
        hide_axes(ax)

        if col == 0:
            weight = 700 if 'alibi' in key else 500
            name = MODEL_NAMES[key]
            name = name.replace('(COCO)', '')
            ax.set_ylabel(name, weight=weight)
        if row == 0:
            ax.set_title(f"({size}, {size})", weight=500)


SAVE = True
if SAVE:
    plt.savefig("saved/S11.pdf", dpi=300, bbox_inches='tight')
    plt.close()


findfont: Failed to find font weight normal, now using 300.


findfont: Failed to find font weight 500, now using 300.
